# Seagrass — train the underwater detector

Fine-tunes YOLOX-Nano on RUOD + TrashCan and exports an ONNX model for the Pi.

**Set the runtime to a GPU first:** Runtime -> Change runtime type -> T4 GPU.
Without one this will not finish — the point of using Colab is the GPU.

The config runs **30 epochs** (`max_epoch` in `configs/yolox_nano_seagrass.py`).
On a free T4 that is roughly **20 minutes** with the `--limit 6000` subsample in
step 3, or **about an hour** on the full ~15k images. Colab disconnects idle
sessions, so keep the tab open, and see the *Resume* cell if it drops.


## 1. Confirm the GPU

Stop here if this reports no GPU. Everything below assumes CUDA, and the
failure otherwise is slow and confusing rather than immediate.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU'
print('torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))


## 2. Install YOLOX

YOLOX pins `onnx-simplifier==0.4.10`, a 2022 release whose last wheel is
cp310. On Colab's newer Python pip therefore falls back to the 18.1 MB source
tarball, and that build fails — `setup.py egg_info did not run successfully`.
It is reported against YOLOX, but YOLOX's own metadata builds fine; the failure
is a *dependency* being resolved after it. Watch the trace: `Preparing metadata
(setup.py) ... done`, an 18.1 MB download, then the error.

So YOLOX is installed with **`--no-deps`**. Nothing is skipped by doing that —
every requirement is already present:

| Requirement | Comes from |
|---|---|
| numpy, torch, torchvision, opencv, tqdm, psutil, tensorboard | preinstalled in Colab |
| loguru, thop, tabulate, ninja, pycocotools, onnx, onnxruntime, onnx-simplifier | the cell below |

onnx-simplifier is used only to tidy the exported ONNX graph, and a current
version does that as well as the pinned one — the pin is age, not a real
constraint.

`--no-build-isolation` stays as well: YOLOX's `setup.py` does `try: import
torch`, which needs the ambient environment to be visible.

> **Do not pin `numpy<2`.** It looks like the fix for YOLOX's age and is not —
> `np.float` went in numpy *1.24*, so 1.26 lacks it too — while Colab's
> opencv 5.x requires numpy>=2 and YOLOX imports opencv. It breaks a real
> dependency to solve nothing, and buries the actual error under conflict
> warnings.

> **Already ran a version that pinned `numpy<2`?** That runtime is in a mixed
> state. Use **Runtime -> Disconnect and delete runtime** and start from the
> top; repairing numpy in place leaves half-initialised modules behind.


In [ ]:
# Everything YOLOX needs that Colab does not already ship.
!pip -q install loguru thop tabulate ninja pycocotools \
                onnx onnxruntime onnx-simplifier

!git clone -q https://github.com/Megvii-BaseDetection/YOLOX /content/YOLOX
%cd /content/YOLOX
# --no-deps skips the onnx-simplifier==0.4.10 pin that cannot build here.
# --no-build-isolation lets setup.py see torch.
# Deliberately NOT -q: if this still fails, the package that failed is named.
!pip install -e . --no-deps --no-build-isolation


### Confirm YOLOX actually imports

The install can report success and still leave YOLOX unimportable. Better to
find out here than forty minutes into a training run.


In [ ]:
import importlib, subprocess, sys
for m in ('yolox', 'torch', 'cv2', 'numpy', 'pycocotools'):
    try:
        mod = importlib.import_module(m)
        print(f'  {m:12s} {getattr(mod, "__version__", "ok")}')
    except Exception as e:
        print(f'  {m:12s} FAILED: {e}')

from yolox.exp import Exp   # the import training actually needs
print('\nYOLOX ready')

# If numpy 2 breaks YOLOX on an `np.float`/`np.int` AttributeError, patch the
# aliases rather than downgrading numpy (which breaks opencv):
#   !grep -rl 'np\.float\b' /content/YOLOX/yolox | xargs -r sed -i 's/np\.float\b/float/g'
#   !grep -rl 'np\.int\b'   /content/YOLOX/yolox | xargs -r sed -i 's/np\.int\b/int/g'


## 3. Get the code and the dataset

Two ways in. **A** re-creates the dataset from source inside Colab and needs
no upload; **B** uses a zip you made locally. A is usually faster, since
Colab downloads far quicker than a home connection uploads.


In [ ]:
!git clone -q https://github.com/S3agrass/Sea-Grass-Drone /content/seagrass
%cd /content/seagrass
!cat training/labels.txt | grep -v '^#'


### Getting the data: download the big half, upload the small half

RUOD is 3.5 GB and TrashCan's `instance_version` is only 206 MB. Colab
downloads far faster than a home connection uploads, so pull RUOD here and
upload only TrashCan — a few minutes instead of an hour, and it avoids needing
a TrashCan download URL I could not verify.

**On your machine, once:**

```bash
cd ~/Documents/datasets
zip -r trashcan_instance.zip trashcan/instance_version   # ~206 MB
```

Put that zip in your Drive, then run both cells below.


In [ ]:
# RUOD: two split tar parts, ~3.5 GB. Untagged GitHub release, so if this 404s
# copy the current asset links from https://github.com/xiaoDetection/RUOD/releases
B='https://github.com/xiaoDetection/RUOD/releases/download/untagged-4f7c7ab75187d68b6449'
!mkdir -p /content/data && cd /content/data && \
  curl -fL -O $B/RUOD.tar.partaa && curl -fL -O $B/RUOD.tar.partab && \
  cat RUOD.tar.part* > RUOD.tar && tar xf RUOD.tar && rm RUOD.tar*
!ls /content/data/RUOD


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Adjust the path if you put the zip somewhere else in Drive.
!mkdir -p /content/data/trashcan
!unzip -q /content/drive/MyDrive/trashcan_instance.zip -d /content/data/
# The zip contains trashcan/instance_version/, so it lands at the path below.
!ls /content/data/trashcan/instance_version


In [ ]:
# Merge into the layout train.sh expects. --dry-run first: read the report
# before writing 3.5GB.
%cd /content/seagrass
!python3 training/scripts/prepare_dataset.py --dry-run --split train \
  --source ruod:/content/data/RUOD/RUOD_ANN/instances_train.json:/content/data/RUOD/RUOD_pic/train \
  --source trashcan:/content/data/trashcan/instance_version/instances_train_trashcan.json:/content/data/trashcan/instance_version/train


In [ ]:
# --limit 6000 keeps a random, seeded 6k of the 15.6k images. 6000/32 = 188
# iters/epoch x the config's 30 epochs is ~20 minutes on a T4, against ~1 hour
# for the full set at the same 30. Random rather than the first N: both datasets
# are ordered, so the head of the list is one dive site in one water, and
# training on that teaches the site instead of the objects. Drop --limit for the
# full run later.
!python3 training/scripts/prepare_dataset.py --split train --symlink --limit 6000 \
  --source ruod:/content/data/RUOD/RUOD_ANN/instances_train.json:/content/data/RUOD/RUOD_pic/train \
  --source trashcan:/content/data/trashcan/instance_version/instances_train_trashcan.json:/content/data/trashcan/instance_version/train
# Val stays smaller anyway; 2000 is plenty to measure mAP.
!python3 training/scripts/prepare_dataset.py --split val --symlink --limit 2000 \
  --source ruod:/content/data/RUOD/RUOD_ANN/instances_test.json:/content/data/RUOD/RUOD_pic/test \
  --source trashcan:/content/data/trashcan/instance_version/instances_val_trashcan.json:/content/data/trashcan/instance_version/val


## 4. Check the dataset before spending hours on it

A missing image or an annotation pointing at the wrong id will not stop
training — it degrades the model quietly, and you find out at the end. This
is the same check that passed on the local copy.


In [ ]:
import json, os, collections
root = '/content/seagrass/training/datasets/seagrass_underwater'

# Directories are train2017/val2017 — a COCO naming artefact, not a date.
# YOLOX's COCODataset hardcodes those names and ignores the Exp's self.name,
# so anything else fails at the first batch with 'file named ... not found'.
# Rename in place if an earlier run built 2024 dirs; rebuilding 3.5GB to
# change a directory name would be silly.
for old, new in (('train2024','train2017'), ('val2024','val2017')):
    if os.path.isdir(f'{root}/{old}') and not os.path.isdir(f'{root}/{new}'):
        os.rename(f'{root}/{old}', f'{root}/{new}')
        print(f'renamed {old} -> {new}')

need = [f'{root}/annotations/instances_{s}.json' for s in ('train','val')]
absent = [q for q in need if not os.path.exists(q)]
if absent:
    raise SystemExit(
        'Dataset not built yet — run the step 3 cells first.\n'
        + '\n'.join(f'  missing: {q}' for q in absent))

labels = [l.strip() for l in open('/content/seagrass/training/labels.txt')
          if l.strip() and not l.startswith('#')]
ok = True
for split, d in (('train','train2017'), ('val','val2017')):
    if not os.path.isdir(f'{root}/{d}'):
        print(f'MISSING image dir {root}/{d}'); ok = False; continue
    j = json.load(open(f'{root}/annotations/instances_{split}.json'))
    imgs = {i['id']: i['file_name'] for i in j['images']}
    missing = [f for f in imgs.values() if not os.path.exists(f'{root}/{d}/{f}')]
    orphan = [a for a in j['annotations'] if a['image_id'] not in imgs]
    per = collections.Counter(a['category_id'] for a in j['annotations'])
    empty = [labels[i] for i in range(len(labels)) if per[i] == 0]
    print(f"{split}: {len(imgs)} images in {d}/, {len(j['annotations'])} anns, "
          f'missing={len(missing)} orphan={len(orphan)}')
    if missing: print('   e.g.', missing[:3])
    if empty: print('   classes with NO examples:', empty)
    ok &= not missing and not orphan
assert len(labels) == 5, f'labels.txt has {len(labels)} entries, config expects 5'
print('\nDATASET OK' if ok else '\nPROBLEMS ABOVE — fix before training')


## 5. Pretrained weights

Fine-tuning from COCO rather than from scratch. On ~15k images that is the
difference between a usable model and one that never converges.


In [ ]:
!curl -fL -o /content/seagrass/training/yolox_nano.pth \
  https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth
!ls -la /content/seagrass/training/yolox_nano.pth


## 6. Train

`BATCH=32` suits a T4's 16GB at 416px; drop to 16 if you hit OOM. Raise
`max_epoch` in the config for a better model at proportional cost.

Progress prints per iteration. Losses should fall steadily; if they go NaN,
lower the batch size or disable fp16.


In [ ]:
%cd /content/seagrass/training
# NOT piped through `tail`. Truncating this output hides a run that died in
# the first minute and makes it look like training happened — which is exactly
# how you end up at the export step wondering where the checkpoint went.
!BATCH=32 DEVICES=1 FP16=1 bash scripts/train.sh


### Did training actually produce a checkpoint?

`best_ckpt.pth` is only written after an evaluation pass, which happens every
`eval_interval` epochs — **3** in this config. A run that died in the first
couple of epochs leaves `latest_ckpt.pth` but no `best_ckpt.pth`.


In [ ]:
import os, glob
d = '/content/seagrass/training/YOLOX_outputs/yolox_nano_seagrass'
if not os.path.isdir(d):
    raise SystemExit(
        f'No output directory at {d}\n'
        'Training did not get far enough to write anything. Scroll up through\n'
        'the training cell output for the real error — common ones are CUDA OOM\n'
        '(lower BATCH), or the dataset check above never having passed.')
for f in sorted(glob.glob(d + '/*.pth')):
    print(f'  {os.path.basename(f):20s} {os.path.getsize(f)/1e6:7.1f} MB')

best, latest = f'{d}/best_ckpt.pth', f'{d}/latest_ckpt.pth'
if os.path.exists(best):
    CKPT = best; print('\nusing best_ckpt.pth')
elif os.path.exists(latest):
    CKPT = latest
    print('\nNo best_ckpt.pth — training ran but never reached an eval pass.')
    print('Falling back to latest_ckpt.pth. Usable, but it is whatever the last')
    print('epoch produced rather than the best-scoring one.')
else:
    raise SystemExit(f'No .pth checkpoints in {d} — training produced nothing.')
print('CKPT =', CKPT)


### Resume after a disconnect

Colab drops long sessions. This picks up from the last checkpoint instead of
restarting from epoch 0.


In [ ]:
# !cd /content/seagrass/training && \
#   BATCH=32 RESUME=YOLOX_outputs/yolox_nano_seagrass/latest_ckpt.pth bash scripts/train.sh


## 7. Export to ONNX and download

This is the file the Pi runs. Copy it to `server/vision/models/` and point
`DETECT_MODEL` at it — nothing else on the drone changes.

**Also copy `training/labels.txt` across.** The model outputs bare class
indices; `labels.txt` is what turns index 4 into `rov`. A stale labels file on
the Pi — the old 12-class one, say — means every box gets confidently
mislabelled, and nothing about the output looks wrong.


In [ ]:
%cd /content/seagrass/training
# CKPT comes from the cell above, so this uses best_ckpt when it exists and
# falls back to latest_ckpt when it does not.
!bash scripts/export_onnx.sh "{CKPT}"

import onnxruntime as ort
s = ort.InferenceSession('/content/seagrass/server/vision/models/seagrass_nano.onnx',
                         providers=['CPUExecutionProvider'])
i, o = s.get_inputs()[0], s.get_outputs()[0]
print('input ', i.name, i.shape)
print('output', o.name, o.shape)
# Expect [1, 3549, 10] at 416px with 5 classes: 3549 anchors, 4 box + 1 obj + 5.
# A different last dimension means num_classes and labels.txt disagree.
assert o.shape[-1] == 10, f'expected 10 (4+1+5), got {o.shape[-1]}'
print('\nlooks right')


In [ ]:
from google.colab import files
files.download('/content/seagrass/server/vision/models/seagrass_nano.onnx')
files.download('/content/seagrass/training/labels.txt')


## 8. Deploy on the Pi

```bash
scp seagrass_nano.onnx pi@seagrass.local:~/Sea-Grass-Drone/server/vision/models/
scp labels.txt        pi@seagrass.local:~/Sea-Grass-Drone/server/vision/models/seagrass.txt

# in ~/.seagrass-env
DETECT_MODEL=/home/pi/Sea-Grass-Drone/server/vision/models/seagrass_nano.onnx
DETECT_LABELS=/home/pi/Sea-Grass-Drone/server/vision/models/seagrass.txt

sudo systemctl restart drone-server
```

Check it before trusting it — this prints straight to the terminal, where the
server would route it to the log:

```bash
DETECT_MODEL=... DETECT_LABELS=... python3 server/vision/detector.py
```

Leave `DETECT_UNDERWATER=1` this time. The filter corrects the blue-green
cast, and the model was trained on underwater images that have it.
